# Dirac-3S QPLIB Benchmarking with eqc_direct

In [1]:
from eqc_direct.client import EqcClient
from eqc_direct.utils import *
import numpy as np
import time
import os

ip_address="172.18.15.110"
client = EqcClient(ip_address=ip_address) 

### Set up dirac-3S solver

In [2]:
def direct_dirac3(indices, coefficients, relaxation_schedule, sum_constraint, num_samples, ip_address):
    eqc_client = EqcClient(ip_address=ip_address) #S9 "172.18.41.45", #S3 "172.18.41.173"
    lock_id, start_ts, end_ts=eqc_client.wait_for_lock()
    try:
        result_dict = eqc_client.solve_sum_constrained(
            lock_id=lock_id,
            poly_indices = indices,
            poly_coefficients = coefficients,
            relaxation_schedule = relaxation_schedule,
            sum_constraint = sum_constraint,
            num_samples = num_samples,
        )
        total_time = np.array(result_dict["preprocessing_time"])+np.array(result_dict["postprocessing_time"])+np.array(result_dict["runtime"])
        print(f"Total execution time(s):{total_time}")
    finally:
        # release lock when finished using the device
        lock_release_out = eqc_client.release_lock(lock_id=lock_id)
    return result_dict

### load instance function

In [3]:
def read_qplib(path):
    with open(path) as fh:
        toks = [t for t in (line.split("#")[0].strip() for line in fh) if t]

    pos = 0

    def nxt():
        nonlocal pos
        val = toks[pos]
        pos += 1
        return val

    def defaulted(length):
        """Read a `default value` / `number of non-defaults` / entries block."""
        arr = np.full(length, float(nxt()))
        for _ in range(int(nxt())):
            i, v = nxt().split()
            arr[int(i) - 1] = float(v)
        return arr

    name = nxt()
    probtype = nxt()
    sense = nxt().lower()
    n = int(nxt())
    m = int(nxt())

    # quadratic terms of the objective (each unordered pair listed once)
    Q = np.zeros((n, n), dtype=np.float32)
    for _ in range(int(nxt())):
        i, j, v = nxt().split()
        i, j, v = int(i) - 1, int(j) - 1, float(v)
        if i == j:
            Q[i, i] = v
        else:
            Q[i, j] = Q[j, i] = 0.5 * v   # split the pair coefficient over both triangles

    b = defaulted(n)            # linear terms of the objective
    obj_const = float(nxt())

    # linear terms of the constraints
    A = np.zeros((m, n))
    for _ in range(int(nxt())):
        k, i, v = nxt().split()
        A[int(k) - 1, int(i) - 1] = float(v)

    inf = float(nxt())
    lhs, rhs = defaulted(m), defaulted(m)
    lb, ub = defaulted(n), defaulted(n)

    return dict(name=name, type=probtype, sense=sense, n=n, m=m,
                Q=Q, b=b, obj_const=obj_const,
                A=A, lhs=lhs, rhs=rhs, lb=lb, ub=ub, inf=inf)


### solve using dirac-3S

In [4]:
inst_dir = "Instances/"
inst_name = "QPLIB_2761.qplib"
inst = read_qplib(os.path.join(inst_dir, inst_name))
n, Q, b = inst["n"], inst["Q"], inst["b"]
M = 0.5 * Q
c = b.copy()

poly_indices, poly_coefficients = convert_hamiltonian_to_poly_format(
    linear_terms=c,
    quadratic_terms=M.copy(),   # the util mutates its input
)

In [5]:
def objective(x):
    """QPLIB objective, same convention as the poly submitted to Dirac-3."""
    x = np.asarray(x, dtype=float)
    return float(x @ M @ x + c @ x) + inst["obj_const"]

sum_constraint = float(inst["rhs"][0])   # sum_i x_i = 1
relaxation_schedule = 1
num_samples = 100


poly_indices, poly_coefficients = convert_hamiltonian_to_poly_format(
                    linear_terms=c,
                    quadratic_terms=M.copy(),   # the util mutates its input
                )

dirac_start = time.time()
print("Submitting job to Dirac-3S")
response = direct_dirac3(indices=poly_indices,
                        coefficients=poly_coefficients,
                        relaxation_schedule=relaxation_schedule,
                        sum_constraint=sum_constraint,
                        ip_address=ip_address,
                        num_samples=num_samples)
print(f"Dirac-3 Run complete. Response received.")
energies = response['energy']
solutions = response['solution']
runtimes = response['runtime']
preprocessing_time = response['preprocessing_time']
postprocessing_time = response['postprocessing_time']

print(f"Dirac-3 all samples energies:{[round(e, 2) for e in energies]}")
print(f"Dirac-3 all samples runtimes:{[round(t, 2) for t in runtimes]}")



Submitting job to Dirac-3S
Total execution time(s):[2.60927448 3.92017277 3.66367798 5.65877393 3.50433762 3.31685099
 4.9969642  2.65017603 2.94848334 3.96039353 4.76025672 5.69540859
 3.56813195 4.78678866 3.77307181 3.79501493 3.80302066 3.06004788
 3.61366897 3.79009187 3.36575746 2.88107067 3.71976098 4.5675675
 3.07030962 4.16581185 4.04716632 4.73760869 3.84494819 2.83097038
 3.98634813 4.67104459 3.73876965 3.85616669 2.68340677 4.18464988
 3.98370418 4.75931722 3.90973101 4.86702825 4.46201827 2.94292931
 4.27756047 3.44825469 3.7916764  2.74540909 3.15322383 4.18542743
 2.83082731 3.97312964 2.9065745  4.4457614  3.55981524 3.19108973
 3.93139399 3.77664767 3.94058345 2.63672721 3.83822486 3.5642059
 3.71930424 3.01746587 2.71147231 5.44961055 3.03422832 3.50443695
 3.14953573 4.63366989 3.4780005  4.37385515 4.02699331 4.06825119
 3.804987   2.96381695 3.53761173 3.83295924 3.04909765 3.63119259
 4.00034818 3.93047291 4.8892151  2.73325073 2.97676449 4.75357767
 3.2108481  3

In [6]:
best = min(zip(energies, solutions, runtimes), key=lambda x: x[0])
dirac_energy, best_solution, best_runtime = best


print(f"Eenergy value obtained by Dirac-3:{dirac_energy}")
print(f"Solution sum:{sum(best_solution)} (target {sum_constraint}), support:{np.count_nonzero(best_solution)}")
print(f"Preprocessing Time:{preprocessing_time}")
print(f"Time taken for this run:{best_runtime}")

Eenergy value obtained by Dirac-3:0.0010485
Solution sum:1.0000001 (target 1.0), support:2
Preprocessing Time:0.22956259548664093
Time taken for this run:3.384004592895508
